In [1]:
!pip install -q -U transformers accelerate bitsandbytes pandas huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 52.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.

In [15]:
#Optional: Free up GPU memory if you plan to change MODEL_ID and run again
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

In [19]:
def evaluate_local_model(crassarray, model, tokenizer):
    col_pred = f"{MODEL_ID.split('/')[-1]}_Prediction"
    col_verdict = f"{MODEL_ID.split('/')[-1]}_Verdict"
    crassarray[col_pred] = ""
    crassarray[col_verdict] = ""

    correct_count = 0
    total_rows = len(crassarray)

    for i in range(total_rows):
        row = crassarray.iloc[i]

        # Extract options and shuffle to prevent positional bias
        options = [ans for ans in [row['CorrectAnswer'], row['Answer1'], row['Answer2']] if pd.notna(ans) and str(ans).strip() != ""]
        random.shuffle(options)
        correct_idx = options.index(row['CorrectAnswer']) + 1
        options_str = "\n".join([f"{idx+1}. {opt}" for idx, opt in enumerate(options)])

        prompt = (
            f"Given the premise, answer the question by selecting the most logical option.\n\n"
            f"Premise: {row['Premise']}\n"
            f"Question: {row['QCC']}\n\n"
            f"Options:\n{options_str}\n\n"
            f"Reply with ONLY the number of the correct option (e.g., 1, 2, or 3)."
        )

        # Format the prompt using the model's native chat template
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        # Tokenize and move to GPU
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

        # Generate the response locally
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512, # We only need the number
                do_sample=False,  # Deterministic output (temperature=0 equivalent)
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated tokens (ignore the prompt)
        input_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[0][input_length:]
        generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        if "</think>" in generated_text:
            final_answer = generated_text.split("</think>")[-1].strip()
        else:
            final_answer = generated_text.strip()

        crassarray.at[i, col_pred] = final_answer

        # Validate the extracted digit
        if final_answer and final_answer[0] == str(correct_idx):
            crassarray.at[i, col_verdict] = "TRUE"
            correct_count += 1
        else:
            crassarray.at[i, col_verdict] = "FALSE"

        if i % 20 == 0 and i > 0:
            print(f"Processed {i}/{total_rows}...")

    accuracy = correct_count / total_rows
    print(f"\nFinal Accuracy for {MODEL_ID}: {accuracy:.2%}")
    return crassarray

In [20]:
import torch
import pandas as pd
import random
import gc
from getpass import getpass
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Authenticate to download the weights
print("Please enter your Hugging Face Access Token to download the model weights:")
login(token=getpass())

# 2. Select the model you want to deploy locally in this session
# Options:
# "mistralai/Mistral-7B-Instruct-v0.3"
# "Qwen/Qwen3-8B"
# "meta-llama/Llama-3.1-8B-Instruct"
MODEL_ID = "Qwen/Qwen3-8B"

print(f"\n--- Loading {MODEL_ID} into Colab GPU (4-bit) ---")

# 3. Configure 4-bit quantization to fit the T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto", # Automatically assigns to GPU
    low_cpu_mem_usage=True
)
print("Model loaded successfully!")

# 4. Evaluation Function
import json

def evaluate_local_model_cot(crassarray, model, tokenizer):
    col_pred = f"{MODEL_ID.split('/')[-1]}_CoT_Prediction"
    col_verdict = f"{MODEL_ID.split('/')[-1]}_Verdict"
    crassarray[col_pred] = ""
    crassarray[col_verdict] = ""

    correct_count = 0
    total_rows = len(crassarray)

    for i in range(total_rows):
        row = crassarray.iloc[i]

        # Extract options and shuffle
        options = [ans for ans in [row['CorrectAnswer'], row['Answer1'], row['Answer2']] if pd.notna(ans) and str(ans).strip() != ""]
        random.shuffle(options)
        correct_idx = options.index(row['CorrectAnswer']) + 1
        options_str = "\n".join([f"{idx+1}. {opt}" for idx, opt in enumerate(options)])

        # CoT Prompt
        prompt = f"""# Role: Counterfactual Reasoning Analyst

Your task is to identify causal variables and generate causal graphs with edges. All outputs must follow strict JSON formatting for automated saving.

## Context Format
**Premise:** {row['Premise']}
**Questionized Counterfactual Conditional (QCC):** {row['QCC']}

**Options:**
{options_str}

## Task Instructions

1. **Variable Identification**
Extract and classify variables into:
'Exposure (X)': Direct intervention (verb phrase, e.g., "act of opening").
'Outcome (Y)': Final result (state phrase, e.g., "chest opened").
'Covariate (Z)': Pre-existing conditions influencing X/Y.
'Mediator (M)': Mechanism linking X-Y (physical/logical process).

2. **Causal Graph Construction**
Build factual/counterfactual edges following these strict rules:

**Factual Graph Requirements:**
Must include ALL these relationships:
1. Z - X (Covariate influences Exposure)
2. Z - M (Covariate influences Mediator)
3. Z - Y (Covariate influences Outcome)
4. X - M (Exposure affects Mediator)
5. M - Y (Mediator affects Outcome)
6. X - Y (Direct exposure effect)

**Counterfactual Graph Requirements:**
Must include ALL these relationships:
1. Z - M' (Same covariate influence on adjusted Mediator)
2. Z - Y' (Same covariate influence on new Outcome)
3. X' - M' (Modified exposure affects adjusted Mediator)
4. M' - Y' (Adjusted mediator affects new Outcome)
5. X' - Y' (Direct modified exposure effect)

Use prime notation (M', Y') for counterfactual variables. Maintain Z's original relationships.

3. **Final Answer**
Based on your causal graphs, select the most logical option number from the Options list.

CRITICAL INSTRUCTION: You must output ONLY a single, valid JSON object. Do not include any conversational text, section headers, or markdown formatting (do NOT use ```json). Your entire response must start with {{ and end with }}.

Required JSON Schema:
{{
  "Variables": {{...}},
  "Factual_Graph": {{...}},
  "Counterfactual_Graph": {{...}},
  "Final_Answer_Number": 1
}}
"""

        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1500,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        input_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[0][input_length:]
        generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        import re # Make sure to import re at the top of your script

        crassarray.at[i, col_pred] = generated_text

        # ---------------------------------------------------------
        # NEW PARSING LOGIC
        # ---------------------------------------------------------
        try:
            # 1. Try to extract everything between the first { and last }
            json_match = re.search(r'\{.*\}', generated_text, re.DOTALL)

            if json_match:
                clean_json = json_match.group(0)
            else:
                # Fallback if no brackets are found
                clean_json = generated_text.strip()

            # 2. Attempt to parse
            response_data = json.loads(clean_json)
            predicted_ans = str(response_data.get("Final_Answer_Number", ""))

            if predicted_ans == str(correct_idx):
                crassarray.at[i, col_verdict] = "TRUE"
                correct_count += 1
            else:
                crassarray.at[i, col_verdict] = "FALSE"

            print(f"Step {i+1}/{total_rows} | Predicted: Option {predicted_ans} | Correct: Option {correct_idx} | Verdict: {crassarray.at[i, col_verdict]}")

        except json.JSONDecodeError as e:
            crassarray.at[i, col_verdict] = "ERROR_PARSING_JSON"
            print(f"Step {i+1}/{total_rows} | Verdict: ERROR_PARSING_JSON")
            print(f"--- RAW MODEL OUTPUT START ---")
            print(generated_text)
            print(f"--- RAW MODEL OUTPUT END ---")
            print(f"JSON Error: {e}\n")

    accuracy = correct_count / total_rows
    print(f"\nFinal Accuracy for {MODEL_ID}: {accuracy:.2%}")
    return crassarray

# 5. Execute Pipeline
url = 'https://raw.githubusercontent.com/apergo-ai/CRASS-data-set/main/CRASS_FTM_main_data_set.csv'
print("\nDownloading dataset...")
df = pd.read_csv(url, encoding='latin1', sep=";")

print("Starting local evaluation...")
results_df = evaluate_local_model(df, model, tokenizer)

# Save to CSV
output_filename = f"{MODEL_ID.split('/')[-1]}_eval_output.csv"
results_df.to_csv(output_filename, sep=';', encoding='utf-8', index=False)
print(f"\nDone! Results saved to '{output_filename}' in the Colab file explorer.")

# Optional: Free up GPU memory if you plan to change MODEL_ID and run again
# del model
# del tokenizer
# gc.collect()
# torch.cuda.empty_cache()

Please enter your Hugging Face Access Token to download the model weights:
··········

--- Loading Qwen/Qwen3-8B into Colab GPU (4-bit) ---


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model loaded successfully!

Starting local evaluation...
Processed 20/274...
Processed 40/274...
Processed 60/274...
Processed 80/274...
Processed 100/274...
Processed 120/274...
Processed 140/274...
Processed 160/274...
Processed 180/274...
Processed 200/274...
Processed 220/274...
Processed 240/274...
Processed 260/274...

Final Accuracy for Qwen/Qwen3-8B: 76.64%

Done! Results saved to 'Qwen3-8B_eval_output.csv' in the Colab file explorer.


In [ ]:
import pandas as pd
import os

models = [
    "Mistral-7B-Instruct-v0.3",
    "Qwen2.5-7B-Instruct",
    "Llama-3.1-8B-Instruct"
]

# 1. Load and Merge the CSV files
print("--- Loading and Merging Data ---")
dfs = {}
for model in models:
    filename = f"{model}_eval_output.csv"
    if os.path.exists(filename):
        dfs[model] = pd.read_csv(filename, sep=";")
        print(f"Successfully loaded {filename}")
    else:
        print(f"⚠️ WARNING: {filename} not found.")

base_model = list(dfs.keys())[0]
df_merged = dfs[base_model].copy()

for model in list(dfs.keys())[1:]:
    pred_col = f"{model}_Prediction"
    verd_col = f"{model}_Verdict"
    if pred_col in dfs[model].columns and verd_col in dfs[model].columns:
        df_merged[pred_col] = dfs[model][pred_col]
        df_merged[verd_col] = dfs[model][verd_col]

# --- SANITY CHECK ---
print("\n--- Sanity Check: Raw Model Outputs ---")
pred_columns = [f"{m}_Prediction" for m in models]
display(df_merged[pred_columns].head(3))


# 2. Table 1: Overall Accuracy
print("\n--- Table 1: Model Top-1 Accuracy (T1) ---")
accuracy_results = []
for model in dfs.keys():
    verdict_col = f"{model}_Verdict"
    # FIX: Convert the column to string and uppercase before checking
    correct = (df_merged[verdict_col].astype(str).str.upper() == "TRUE").sum()
    total = len(df_merged)
    accuracy = (correct / total) * 100
    accuracy_results.append({"Model": model, "T1 Accuracy (%)": round(accuracy, 2)})

table1_df = pd.DataFrame(accuracy_results)
display(table1_df)


# 3. Performance on "Impossible" Situations
print("\n--- Model Performance on 'Impossible' Situations ---")
impossible_df = df_merged[df_merged['CorrectAnswer'].str.contains("That is not possible", case=False, na=False)]

impossible_results = []
for model in dfs.keys():
    verdict_col = f"{model}_Verdict"
    # FIX: String conversion here too
    correct = (impossible_df[verdict_col].astype(str).str.upper() == "TRUE").sum()
    total = len(impossible_df)
    accuracy = (correct / total) * 100 if total > 0 else 0
    impossible_results.append({"Model": model, "Impossible T1 (%)": round(accuracy, 2)})

impossible_table = pd.DataFrame(impossible_results)
display(impossible_table)


# 4. Model Consensus Analysis
if len(dfs) == 3:
    print("\n--- Model Consensus Analysis ---")

    # FIX: String conversion for consensus
    all_correct = df_merged[
        (df_merged[f"{models[0]}_Verdict"].astype(str).str.upper() == "TRUE") &
        (df_merged[f"{models[1]}_Verdict"].astype(str).str.upper() == "TRUE") &
        (df_merged[f"{models[2]}_Verdict"].astype(str).str.upper() == "TRUE")
    ]

    all_failed = df_merged[
        (df_merged[f"{models[0]}_Verdict"].astype(str).str.upper() == "FALSE") &
        (df_merged[f"{models[1]}_Verdict"].astype(str).str.upper() == "FALSE") &
        (df_merged[f"{models[2]}_Verdict"].astype(str).str.upper() == "FALSE")
    ]

    print(f"Items answered correctly by ALL models: {len(all_correct)} ({len(all_correct)/len(df_merged)*100:.2f}%)")
    print(f"Items failed by ALL models: {len(all_failed)} ({len(all_failed)/len(df_merged)*100:.2f}%)")

    print("\nTop 5 Hardest Questions (Failed by all):")
    display(all_failed[['Premise', 'QCC', 'CorrectAnswer']].head())

--- Loading and Merging Data ---
Successfully loaded Mistral-7B-Instruct-v0.3_eval_output.csv
Successfully loaded Qwen2.5-7B-Instruct_eval_output.csv
Successfully loaded Llama-3.1-8B-Instruct_eval_output.csv

--- Sanity Check: Raw Model Outputs ---


,Mistral-7B-Instruct-v0.3_Prediction,Qwen2.5-7B-Instruct_Prediction,Llama-3.1-8B-Instruct_Prediction
0,2. The treasure,3,1
1,2. The host,2,2
2,2. Without a,3,2



--- Table 1: Model Top-1 Accuracy (T1) ---


,Model,T1 Accuracy (%)
0,Mistral-7B-Instruct-v0.3,78.47
1,Qwen2.5-7B-Instruct,85.77
2,Llama-3.1-8B-Instruct,76.28



--- Model Performance on 'Impossible' Situations ---


,Model,Impossible T1 (%)
0,Mistral-7B-Instruct-v0.3,100.00
1,Qwen2.5-7B-Instruct,100.00
2,Llama-3.1-8B-Instruct,93.75



--- Model Consensus Analysis ---
Items answered correctly by ALL models: 172 (62.77%)
Items failed by ALL models: 17 (6.20%)

Top 5 Hardest Questions (Failed by all):


,Premise,QCC,CorrectAnswer
18,A mortician prepares a corpse.,What would have happened if the mortician had ...,He would have had a delicious dish.
24,A woman swallows potassium cyanide.,What would have happened if the woman had dist...,She would have survived.
25,A woman swallows potassium cyanide.,What would have happened if the woman had watc...,Nothing special would have happened.
31,A man reports a crime.,What would have happened if the man had cleare...,He would have been a detective.
64,A man was sleeping with his mouth wide open.,What would have happened if a spider had come ...,The man would have bitten the spider.
